In [6]:
import os, math, hashlib, requests, warnings
from datetime import datetime, timedelta
from scipy.special import hyp1f1, gamma , factorial, gammaincc , binom , gammainc , poch, hyp2f1

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.special import gamma as gamma_func
from scipy.stats import norm

from skyfield.api import load, EarthSatellite, wgs84

from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score

import pulp
warnings.filterwarnings('ignore')
print("✅ Imports OK")


✅ Imports OK


## Parameters

In [7]:
# ── Ground Station ────────────────────────────────────────────────────────────
STATION = {
    'name': 'Santa Rita do Sapucaí (BR)',
    'lat':   -23.22,
    'lon':   -45.89,
    'alt_m':  650,
}

# ── Time windows ──────────────────────────────────────────────────────────────
HISTORY_DAYS = 90
TRAIN_DAYS   = 60
FUTURE_DAYS  = 10

# ── Satellites ────────────────────────────────────────────────────────────────
TLE_LOCAL_CSV     = 'tle_active.csv'
SAT_NAME_CONTAINS = 'STARLINK'
N_SATELLITES      = 15

# ── Physics-Informed Proxy Label Parameters ───────────────────────────────────
# Ali Abdi channel model — polynomial coefficients (Table IV, DtS-IoT paper)
# b(θ) = b3·θ³ + b2·θ² + b1·θ + b0  (same for m, Omega)
ALI_ABDI_COEFFS = {
    'b': [ 3.2710e-2, -2.1344e-4,  5.5784e-6, -4.7943e-8],   # [b0,b1,b2,b3]
    'm': [ 3.5156,    -1.5973e-1,  5.8533e-4,  6.3739e-5],
    'O': [-1.4864,     1.2702e-1, -2.3798e-3,  1.4428e-5],
}

# ITU-R P.838 coefficients — Ku-band downlink (12 GHz)
'''ITU_P838_KR    = #0.0188   # k coefficient
ITU_P838_AR    = 1.217    # alpha coefficient'''

ITU_P838_KR = 0.0000923 #0.02421 #0.0000923
ITU_P838_AR = 1.003 # 1.1516 #1.003

RAIN_HEIGHT_KM = 3.0      # h_R — effective rain height (ITU-R P.839)

# Link budget (from DtS-IoT paper, Table V — adapted to Ku-band)
FC_GHZ         = 2.0 #12.0 #2.0     # Carrier frequency [GHz]
PS_DBM         = 30.0 #55.0 #30.0     # Transmit power [dBm]
DI_DB          = 5.19 # 35.0 #5.19     # IoT device antenna gain [dB]
DN_DB          = 6.97  #38.0 #6.97     # Satellite antenna gain [dB]
SNR_TH_DB      = 1.0 #10.0 #1.0      # SNR threshold [dB]
T_NOISE_K      = 290.0    # Receiver noise temperature [K]
BANDWIDTH_HZ   = 180e3    # Bandwidth [Hz]
H_ORBIT_KM     = 550.0    # Starlink orbital altitude [km]

# Label noise on training set (to simulate real channel uncertainty)
LABEL_FLIP_SIGMA = 0.05   # std of Gaussian noise added to p_proxy ∈ [0,1]

# ── Optimization ──────────────────────────────────────────────────────────────
BITRATE_KBPS          = 1000.0
MIN_SETUP_S           = 120
SETUP_COST_KB         = 0.0
MAX_TOTAL_SCHEDULED_S = 3600

# ── ML ────────────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
rng = np.random.RandomState(RANDOM_STATE)

# ── Cache ─────────────────────────────────────────────────────────────────────
FORCE_DOWNLOAD = False
CACHE_DIR      = 'cache'

# ── Generalization stations ───────────────────────────────────────────────────
STATIONS_GLOBAL = [
    {'name': 'Sta. Rita Sapucaí (BR)', 'lat': -23.22, 'lon': -45.89, 'alt_m':  650, 'continent': 'South America'},
    {'name': 'São Paulo (BR)',          'lat': -23.55, 'lon': -46.63, 'alt_m':  760, 'continent': 'South America'},
    {'name': 'Toronto (CA)',            'lat':  43.65, 'lon': -79.38, 'alt_m':   76, 'continent': 'North America'},
    {'name': 'Quebec (CA)',             'lat':  46.81, 'lon': -71.21, 'alt_m':   91, 'continent': 'North America'},
    {'name': 'London (UK)',             'lat':  51.51, 'lon':  -0.13, 'alt_m':   11, 'continent': 'Europe'},
    {'name': 'Madrid (ES)',             'lat':  40.42, 'lon':  -3.70, 'alt_m':  667, 'continent': 'Europe'},
    {'name': 'Nairobi (KE)',            'lat':  -1.29, 'lon':  36.82, 'alt_m': 1795, 'continent': 'Africa'},
    {'name': 'Cotonou (BJ)',            'lat':   6.37, 'lon':   2.42, 'alt_m':    9, 'continent': 'Africa'},
    {'name': 'Mumbai (IN)',             'lat':  19.08, 'lon':  72.88, 'alt_m':   14, 'continent': 'Asia'},
    {'name': 'Sydney (AU)',             'lat': -33.87, 'lon': 151.21, 'alt_m':   39, 'continent': 'Oceania'},
]

print("✅ Parameters loaded")


✅ Parameters loaded


## Physics-Informed Proxy Label

> **Level 1** — ITU-R P.838 rain attenuation  
> **Level 2** — Link budget with atmospheric losses  
> **Level 3** — Outage probability via Ali Abdi channel model → continuous label $p_i = 1 - \mathcal{O}_n$

In [17]:
# ── LEVEL 1: ITU-R P.838 rain attenuation ─────────────────────────────────────
def compute_rain_attenuation(R_mmh, theta_deg, fc_ghz=FC_GHZ,
                              kR=ITU_P838_KR, aR=ITU_P838_AR,
                              hR_km=RAIN_HEIGHT_KM):
    """Rain attenuation [dB] via ITU-R P.838.

    A_rain = (k_R * R^alpha_R * h_R) / sin(theta)

    Args:
        R_mmh:     Mean precipitation [mm/h]
        theta_deg: Satellite elevation angle [degrees]
        fc_ghz:    Carrier frequency [GHz] — kR,aR already set for 12 GHz
        kR, aR:    ITU-R P.838 frequency-dependent coefficients
        hR_km:     Effective rain height [km] (ITU-R P.839)

    Returns:
        A_rain [dB]
    """
    if R_mmh <= 0.0:
        return 0.0
    theta_rad = math.radians(max(theta_deg, 1.0))   # avoid division by zero
    gamma_R   = kR * (R_mmh ** aR)                  # specific attenuation [dB/km]
    L_rain    = hR_km / math.sin(theta_rad)          # slant path length [km]
    return gamma_R * L_rain


# ── LEVEL 2: Link budget with atmospheric loss ─────────────────────────────────
def compute_link_budget(theta_deg, R_mmh,
                        fc_ghz=FC_GHZ, h_orbit_km=H_ORBIT_KM,
                        Ps_dbm=PS_DBM, Di_db=DI_DB, Dn_db=DN_DB,
                        T_K=T_NOISE_K, B_hz=BANDWIDTH_HZ):
    """Compute SNR [dB] for a pass given elevation and precipitation.

    Returns:
        snr_db:   Instantaneous SNR [dB]
        nu2:      |h_in|^2 linear gain (used in outage threshold phi)
        phi:      Outage threshold phi = (N0/Ps) * SNR_th / nu^2
    """
    c = 3e8
    fc_hz = fc_ghz * 1e9
    # Slant range from orbital geometry
    rE = 6371.0
    r  = rE + h_orbit_km
    dtheta = math.acos((rE / r) * math.cos(math.radians(theta_deg))) - math.radians(theta_deg)
    d_km = math.sqrt(rE**2 + r**2 - 2*rE*r*math.cos(dtheta))
    d_m  = d_km * 1e3

    # Free-space path loss [dB]
    FSPL_db = 20*math.log10(4*math.pi*fc_hz*d_m/c)

    # Rain attenuation [dB]
    A_rain = compute_rain_attenuation(R_mmh, theta_deg, fc_ghz)

    # Total path loss [dB]
    total_loss_db = FSPL_db + A_rain

    # Linear gain nu^2 = Di*Dn / 10^(PL/10)
    nu2 = (10**(Di_db/10) * 10**(Dn_db/10)) / 10**(total_loss_db/10)

    # Noise power N0 = kBT [W]
    k_B = 1.38e-23
    N0  = k_B * T_K * B_hz
    Ps  = 10**((Ps_dbm - 30) / 10)     # dBm → W

    # SNR [dB]
    snr_linear = nu2 * Ps / N0
    snr_db = 10*math.log10(max(snr_linear, 1e-20))

    # Outage threshold phi (Eq. 13 denominator from DtS-IoT paper)
    SNR_th_linear = 10**(SNR_TH_DB/10)
    phi = (N0 / Ps) * SNR_th_linear / max(nu2, 1e-30)
    print(f"Link budget: θ={theta_deg}°, R={R_mmh}mm/h → SNR={snr_db:.2f}dB, nu^2={nu2:.2e}, phi={phi:.2e}")

    return snr_db, nu2, phi


# ── LEVEL 3: Ali Abdi channel model — outage probability ──────────────────────
def _ali_abdi_params(theta_deg, coeffs=ALI_ABDI_COEFFS):
    """Compute b, m, Omega from elevation angle via polynomial fit (Table IV)."""
    th = float(theta_deg)
    def poly(c):   # c = [c0, c1, c2, c3]
        return c[0] + c[1]*th + c[2]*th**2 + c[3]*th**3
    b = poly(coeffs['b'])
    m = poly(coeffs['m'])
    O = poly(coeffs['O'])
    # Clamp to physically valid ranges
    b = max(b, 1e-6)
    m = max(m, 0.5)
    return b, m, O


def _lower_incomplete_gamma(s, x, n_terms=50):
    """Lower incomplete gamma function gamma(s, x) via series expansion.
    gamma(s, x) = x^s * e^{-x} * sum_{k=0}^{inf} x^k / (s*(s+1)*...*(s+k))
    """
    if x <= 0:
        return 0.0
    total = 0.0
    term  = 1.0 / s
    for k in range(1, n_terms):
        total += term
        term  *= x / (s + k)
    return (x**s) * math.exp(-x) * total


def compute_outage_ali_abdi(theta_deg, R_mmh, n_terms=30):
    """Outage probability O_n(theta, R) via Ali Abdi model (Eq. 13).

    O_n = (2bmK)^m * sum_{k=0}^{inf} (m)_k/k! * (OmegaK)^k * gamma(k+1, phi/2b)

    Args:
        theta_deg: Elevation angle [degrees]
        R_mmh:     Mean precipitation [mm/h]
        n_terms:   Series truncation (30 is sufficient for convergence)

    Returns:
        O_n ∈ [0, 1]  — probability of outage (link failure)
    """
    b, m, Omega = _ali_abdi_params(theta_deg)
    #print(f"Ali Abdi params: θ={theta_deg}° → b={b:.4e}, m={m:.2f}, Omega={Omega:.4e}") 
    K = 1.0 / (2*b*m + Omega)

    # Compute phi from link budget
    _, nu2, phi = compute_link_budget(theta_deg, R_mmh)

    # Prefactor (2bmK)^m
    prefactor = np.abs((2*b*m*K))**m

    # Series sum
    total = 0.0
    #pochhammer = 1.0   # (m)_0 = 1
    for k in range(n_terms):
        '''if k > 0:
            pochhammer *= (m + k - 1) / k   # (m)_k = (m)_{k-1} * (m+k-1)/k
        '''
        pochhammer =  poch(m, k)
        term = (pochhammer/factorial(k))*((Omega/(2*b*m+Omega))** k)
        gamma_term = gammainc(k + 1,phi/(2*b))
        total += term * gamma_term
        #OmegaK_k = (Omega * K) ** k
        #lig      = _lower_incomplete_gamma(k + 1, phi / (2*b))
        #total   += pochhammer * OmegaK_k * lig 
        # Convergence check
        '''if k > 5 and abs(pochhammer * OmegaK_k) < 1e-10:
            break'''  
    O_n = prefactor * total
    return float(np.clip(O_n, 0.0, 1.0))


# ── Physics proxy label ────────────────────────────────────────────────────────
def physics_proxy_label(theta_deg, R_mmh, noise_sigma=0.0):
    """Physics-informed continuous proxy label.

    p_i = 1 - O_n(theta_i, R_i)  ∈ [0, 1]

    p_i = 1 → certain success  (good channel, no rain)
    p_i = 0 → certain failure  (deep fading or heavy rain)

    Args:
        theta_deg:   Peak elevation angle of pass [degrees]
        R_mmh:       Mean precipitation during pass [mm/h]
        noise_sigma: Gaussian noise std added to simulate label uncertainty
                     (used only on training set, 0.0 on test set)

    Returns:
        p_i ∈ [0, 1]
    """
    p = 1.0 - compute_outage_ali_abdi(theta_deg, R_mmh)
    if noise_sigma > 0:
        p += rng.normal(0, noise_sigma)
    return float(np.clip(p, 0.0, 1.0))


# ── Quick sanity check ────────────────────────────────────────────────────────
print("Physics proxy label — sanity check:")
print("─" * 60)
test_cases = [
    (80, 0.0,  "High elev, no rain    → expect p ≈ 1.0"),
    (80, 0.6,  "High elev, light rain → binary=0, physics compensates"),
    (21, 0.4,  "Low elev, light rain  → binary=1, physics shows risk"),
    (21, 5.0,  "Low elev, heavy rain  → expect p ≈ 0.0"),
    (45, 0.5,  "Mid elev, at threshold→ intermediate p"),
    (30, 2.0,  "Low elev, med rain    → degraded link"),
    (19, 0.0,  "Ultra-Low elev, no rain     → expect p ≈ 1.0"),
]
for th, R, desc in test_cases:
    O  = compute_outage_ali_abdi(th, R)
    p  = 1.0 - O
    bin_label = 1 if (th >= 20 and R <= 0.5) else 0
    print(f" θ={th:2}°  R={R:.1f}mm/h  O_n={O:.3f}  p_proxy={p:.3f}  binary={bin_label}  | {desc}")


Physics proxy label — sanity check:
────────────────────────────────────────────────────────────
Link budget: θ=80°, R=0.0mm/h → SNR=10.19dB, nu^2=7.53e-15, phi=1.20e-01
 θ=80°  R=0.0mm/h  O_n=0.001  p_proxy=0.999  binary=1  | High elev, no rain    → expect p ≈ 1.0
Link budget: θ=80°, R=0.6mm/h → SNR=10.19dB, nu^2=7.53e-15, phi=1.20e-01
 θ=80°  R=0.6mm/h  O_n=0.001  p_proxy=0.999  binary=0  | High elev, light rain → binary=0, physics compensates
Link budget: θ=21°, R=0.4mm/h → SNR=3.14dB, nu^2=1.49e-15, phi=6.10e-01
 θ=21°  R=0.4mm/h  O_n=0.847  p_proxy=0.153  binary=1  | Low elev, light rain  → binary=1, physics shows risk
Link budget: θ=21°, R=5.0mm/h → SNR=3.14dB, nu^2=1.48e-15, phi=6.11e-01
 θ=21°  R=5.0mm/h  O_n=0.847  p_proxy=0.153  binary=0  | Low elev, heavy rain  → expect p ≈ 0.0
Link budget: θ=45°, R=0.5mm/h → SNR=7.63dB, nu^2=4.17e-15, phi=2.17e-01
 θ=45°  R=0.5mm/h  O_n=0.087  p_proxy=0.913  binary=1  | Mid elev, at threshold→ intermediate p
Link budget: θ=30°, R=2.0mm/h → 

In [3]:
import itur
import itur.models.itu618 as itu531

def compute_S4_itur(lat_deg, lon_deg, fc_ghz, elevation_deg, 
                    local_hour, doy, p_percent=1.0):
    """S4 via biblioteca itur (ITU-R P.531).
    
    Args:
        lat_deg, lon_deg : coordenadas da estação
        fc_ghz           : frequência [GHz]
        elevation_deg    : ângulo de elevação do satélite
        local_hour       : hora local (0-23)
        doy              : dia do ano (1-365)
        p_percent        : percentual de tempo excedido
    
    Returns:
        S4 : índice de cintilação na frequência fc_ghz
    """
    import astropy.units as u
    
    # Ângulo zenital = 90° - elevação
    zenith_deg = 90.0 - elevation_deg
    
    # S4 via ITU-R P.531
    # A biblioteca retorna S4 na frequência especificada
    S4 = itu531.scintillation_attenuation_sigma(
        lat=lat_deg * u.deg,
        lon=lon_deg * u.deg,
        f=fc_ghz * u.GHz,
        el=elevation_deg * u.deg,
        p=p_percent,
        D=1.0 * u.m,   # antena omnidirecional
        eta=1.0
    )
    return float(S4.value)

In [ ]:
compute_S4_itur(lat_deg=STATION['lat'], lon_deg=STATION['lon'], fc_ghz=2.0, elevation_deg=20,  local_hour=21, doy=80, p_percent=1.0)




UnitConversionError: Can only apply 'remainder' function to dimensionless quantities when other argument is not a quantity (unless the latter is all zero/infinity/nan).

In [10]:
# ── REMOVER a função compute_S4_itur que não funciona ─────────────────────
# itur NÃO implementa ITU-R P.531 (ionosférica)
# itur implementa ITU-R P.618 (troposférica) — diferente e errado para 2 GHz

# ── USAR a função correta baseada diretamente na ITU-R P.531-16 ───────────

def compute_S4_p531(lat_deg, local_hour, doy,
                    solar_activity='moderate',
                    fc_ghz=2.0, f_ref_ghz=1.575):
    """S4 ionosférico via ITU-R P.531-16 — modelo climatológico.

    Baseado nas relações qualitativas da norma:
      Seção 5.1  : S4 ∝ f^(-1.5) — escala de frequência
      Seção 5.2.1: atividade máxima em ±20° do equador magnético
      Seção 5.2.2: pico pós-pôr do sol (20h-23h LT), equinócio vernal

    Nota: ITU-R P.531-16 não fornece fórmula fechada para S4(lat,hora,doy).
    Este modelo usa as relações qualitativas da norma com parametrização
    climatológica. Para S4 real, usar dados ISMR/INPE.

    Args:
        lat_deg       : latitude [graus]
        local_hour    : hora local (0-23)
        doy           : dia do ano (1-365)
        solar_activity: 'low' | 'moderate' | 'high'
        fc_ghz        : frequência de operação [GHz]
        f_ref_ghz     : frequência de referência = GPS L1 = 1.575 GHz

    Returns:
        S4_iono : índice ionosférico (independente de frequência)
        S4_fc   : S4 escalado para fc_ghz
        A_scint : atenuação [dB] (Eq.8 + Seção 5.8, ITU-R P.531-16)
    """
    import numpy as np, math

    S4_base = {'low': 0.25, 'moderate': 0.55, 'high': 0.85}[solar_activity]

    # Fator geográfico (Seção 5.2.1)
    lat_factor    = max(0.0, 1.0 - abs(lat_deg + 11.0) / 20.0)

    # Fator temporal — pico 20h-23h LT (Seção 5.2.2)
    hour_factor   = np.exp(-0.5 * ((local_hour - 21.5) / 1.5)**2)

    # Fator sazonal — equinócio vernal/outonal (Seção 5.2.2)
    s_vernal      = 0.5 * (1 + np.cos(2*np.pi*(doy - 80)  / 365))
    s_autumnal    = 0.5 * (1 + np.cos(2*np.pi*(doy - 265) / 365))
    season_factor = max(s_vernal, s_autumnal * 0.7)

    S4_iono = float(np.clip(
        S4_base * lat_factor * hour_factor * season_factor, 0.0, 1.0))

    # Escala de frequência (Seção 5.1: S4 ∝ f^-1.5)
    S4_fc = float(np.clip(
        S4_iono * (fc_ghz / f_ref_ghz)**(-1.5), 0.0, 1.0))

    # Atenuação (Eq.8 + Passo 4, Seção 5.8)
    A_scint = 0.0
    if S4_fc >= 0.01:
        P_fluc  = 27.5 * (S4_fc**1.26) * (fc_ghz / 4.0)**(-1.5)
        A_scint = float(P_fluc / math.sqrt(2))

    return S4_iono, S4_fc, A_scint


# ── Teste equivalente ao que você tentou ──────────────────────────────────
S4_iono, S4_fc, A_scint = compute_S4_p531(
    lat_deg        = STATION['lat'],   # -23.22
    local_hour     = 21,
    doy            = 80,
    solar_activity = 'moderate',
    fc_ghz         = 2.0
)

print(f"S4_iono  = {S4_iono:.4f}  (intensidade ionosférica)")
print(f"S4_fc    = {S4_fc:.4f}  (S4 escalado para {FC_GHZ} GHz)")
print(f"A_scint  = {A_scint:.2f} dB")
# Resultado esperado:
# S4_iono  = 0.2019
# S4_fc    = 0.1413
# A_scint  = 4.68 dB

S4_iono  = 0.2024  (intensidade ionosférica)
S4_fc    = 0.1414  (S4 escalado para 2.0 GHz)
A_scint  = 4.68 dB


## Utility Functions

In [ ]:
# ── Cache ─────────────────────────────────────────────────────────────────────
def _cache_path(name, ext='csv'):
    os.makedirs(CACHE_DIR, exist_ok=True)
    return os.path.join(CACHE_DIR, f'{name}.{ext}')

def _cache_exists(name, ext='csv'):
    return os.path.exists(_cache_path(name, ext)) and not FORCE_DOWNLOAD

def _cache_key(lat, lon, start_dt, end_dt, kind):
    s = f"{lat:.4f}_{lon:.4f}_{start_dt.date()}_{end_dt.date()}_{kind}"
    return hashlib.md5(s.encode()).hexdigest()[:12]

# ── TLE / Satellites ───────────────────────────────────────────────────────────
def load_satellites_from_csv(csv_path, name_contains=SAT_NAME_CONTAINS,
                              n_satellites=N_SATELLITES):
    df     = pd.read_csv(csv_path)
    subset = df[df['Satellite'].str.contains(name_contains, case=False)].head(n_satellites)
    ts     = load.timescale()
    sats   = []
    for _, row in subset.iterrows():
        sat = EarthSatellite(row['Line1'], row['Line2'], row['Satellite'], ts)
        sats.append({'name': row['Satellite'], 'sat': sat})
    print(f'  {len(sats)} satellites loaded')
    return sats

def get_satellites():
    cached_tle = _cache_path('tle_active')
    if not FORCE_DOWNLOAD and os.path.exists(TLE_LOCAL_CSV):
        print(f'  TLE: using local file ({TLE_LOCAL_CSV})')
        return load_satellites_from_csv(TLE_LOCAL_CSV)
    if _cache_exists('tle_active'):
        print(f'  TLE: using cache')
        return load_satellites_from_csv(cached_tle)
    print('  TLE: downloading from Celestrak...')
    import io
    r = requests.get('https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=csv',
                     timeout=120)
    r.raise_for_status()
    df_tle = pd.read_csv(io.StringIO(r.text))
    df_tle.to_csv(cached_tle, index=False)
    return load_satellites_from_csv(cached_tle)

# ── Orbital passes ─────────────────────────────────────────────────────────────
def compute_passes_all_sats(sat_list, station, start_dt, end_dt,
                             min_elevation_deg=5.0):
    ts = load.timescale()
    times, cur = [], start_dt
    while cur <= end_dt:
        times.append(cur)
        cur += timedelta(minutes=1)
    t_ts = ts.utc([t.year for t in times], [t.month for t in times],
                  [t.day  for t in times], [t.hour  for t in times],
                  [t.minute for t in times], [t.second for t in times])
    sp = wgs84.latlon(station['lat'], station['lon'], station['alt_m'])
    all_passes, global_idx = [], 0
    for entry in sat_list:
        alt, _, _ = (entry['sat'] - sp).at(t_ts).altaz()
        elevs, visible = alt.degrees, alt.degrees > 0
        i = 0
        while i < len(visible):
            if visible[i]:
                j = i
                while j+1 < len(visible) and visible[j+1]:
                    j += 1
                elv   = elevs[i:j+1]
                max_e = float(np.max(elv))
                dur   = (times[j] - times[i]).total_seconds()
                if max_e >= min_elevation_deg and dur >= 10:
                    all_passes.append({
                        'pass_id':            f'pass_{global_idx}',
                        'satellite':          entry['name'],
                        'start_utc':          times[i],
                        'end_utc':            times[j],
                        'duration_s':         dur,
                        'max_elevation_deg':  max_e,
                        'elevation_mean_deg': float(np.mean(elv)),
                        'elevation_std_deg':  float(np.std(elv)),
                    })
                    global_idx += 1
                i = j+1
            else:
                i += 1
    return pd.DataFrame(all_passes).sort_values('start_utc').reset_index(drop=True)

# ── Meteorological data ────────────────────────────────────────────────────────
def _fetch_meteo(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    j   = r.json()
    key = 'cloudcover' if 'cloudcover' in j.get('hourly', {}) else 'cloud_cover'
    n   = len(j['hourly']['time'])
    df  = pd.DataFrame({
        'time':          j['hourly']['time'],
        'precipitation': j['hourly']['precipitation'],
        'cloudcover':    j['hourly'].get(key, [0]*n),
    })
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    return df

def fetch_open_meteo_forecast(lat, lon, start_dt, end_dt):
    key  = _cache_key(lat, lon, start_dt, end_dt, 'forecast')
    path = _cache_path(f'meteo_forecast_{key}')
    if _cache_exists(f'meteo_forecast_{key}'):
        df = pd.read_csv(path, parse_dates=['time'])
        df.set_index('time', inplace=True)
        return df
    df = _fetch_meteo(
        f'https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}'
        f'&hourly=precipitation,cloudcover'
        f'&start_date={start_dt.date()}&end_date={end_dt.date()}&timezone=UTC')
    df.to_csv(path)
    return df

def fetch_open_meteo_archive(lat, lon, start_dt, end_dt):
    key  = _cache_key(lat, lon, start_dt, end_dt, 'archive')
    path = _cache_path(f'meteo_archive_{key}')
    if _cache_exists(f'meteo_archive_{key}'):
        df = pd.read_csv(path, parse_dates=['time'])
        df.set_index('time', inplace=True)
        return df
    df = _fetch_meteo(
        f'https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}'
        f'&hourly=precipitation,cloudcover'
        f'&start_date={start_dt.date()}&end_date={end_dt.date()}&timezone=UTC')
    df.to_csv(path)
    return df

print("✅ Utility functions loaded")


## Feature Engineering & Physics-Informed Labels

In [ ]:
def features_from_pass(pass_row, meteo_df):
    """Extract 13 features + physics proxy for a single pass."""
    start = pd.to_datetime(pass_row['start_utc'])
    end   = pd.to_datetime(pass_row['end_utc'])
    sel   = meteo_df.loc[(meteo_df.index >= start) & (meteo_df.index <= end)]

    if sel.empty:
        pm = cm = px = cx = 0.0
        pc = 0
    else:
        pm = float(sel['precipitation'].mean())
        cm = float(sel['cloudcover'].mean())
        px = float(sel['precipitation'].max())
        cx = float(sel['cloudcover'].max())
        pc = int((sel['precipitation'] > 0.1).sum())

    # Elevation — from pre-computed columns (robust to CSV load)
    em = float(pass_row.get('elevation_mean_deg', pass_row['max_elevation_deg']))
    es = float(pass_row.get('elevation_std_deg', 0.0))

    mid = start + (end - start) / 2
    h   = mid.hour + mid.minute / 60.0
    doy = mid.timetuple().tm_yday

    return {
        'duration_s':          pass_row['duration_s'],
        'max_elevation_deg':   pass_row['max_elevation_deg'],
        'elevation_mean_deg':  em,
        'elevation_std_deg':   es,
        'precipitation_mean':  pm,
        'precipitation_max':   px,
        'precip_hours_count':  pc,
        'cloudcover_mean':     cm,
        'cloudcover_max':      cx,
        'hour_sin':  math.sin(2*math.pi*h/24),
        'hour_cos':  math.cos(2*math.pi*h/24),
        'doy_sin':   math.sin(2*math.pi*doy/365.25),
        'doy_cos':   math.cos(2*math.pi*doy/365.25),
    }


def build_features_labels(passes_df, meteo_df, noise_sigma=LABEL_FLIP_SIGMA,
                           prefix='p', use_physics=True):
    """Build feature DataFrame with physics-informed continuous labels.

    Args:
        passes_df:     DataFrame of passes
        meteo_df:      Meteorological data (forecast or archive)
        noise_sigma:   Std of Gaussian noise on training labels (0.0 for test)
        prefix:        ID prefix
        use_physics:   If True → p_i = 1 - O_n (continuous, physics-based)
                       If False → binary label (legacy, for comparison)

    Returns:
        DataFrame with features + 'label_success' column ∈ [0,1]
    """
    records = []
    for i, (_, row) in enumerate(passes_df.iterrows()):
        feats = features_from_pass(row, meteo_df)

        if use_physics:
            # Physics-informed continuous label
            p = physics_proxy_label(
                feats['max_elevation_deg'],
                feats['precipitation_mean'],
                noise_sigma=noise_sigma
            )
        else:
            # Legacy binary label (for comparison)
            ok = int(feats['precipitation_mean'] <= 0.5 and
                     feats['max_elevation_deg']  >= 20.0)
            if noise_sigma > 0 and rng.rand() < noise_sigma:
                ok = 1 - ok
            p = float(ok)

        records.append({
            'pass_id':       f'{prefix}_{i}',
            'satellite':     row.get('satellite', ''),
            'start_utc':     row['start_utc'],
            'end_utc':       row['end_utc'],
            **feats,
            'label_success': p,
        })
    return pd.DataFrame(records)


# ── Visualization: physics proxy vs binary proxy ───────────────────────────────
print("Comparing physics proxy vs binary proxy label:")
elevations = np.arange(10, 85, 5)
rain_vals  = [0.0, 0.5, 1.0, 2.0, 5.0]
colors     = ['#1D9E75','#3B8BD4','#E24B4A','#BA7517','#9B59B6']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for R, col in zip(rain_vals, colors):
    p_phys = [physics_proxy_label(th, R) for th in elevations]
    p_bin  = [1.0 if (th >= 20 and R <= 0.5) else 0.0 for th in elevations]
    axes[0].plot(elevations, p_phys, 'o-', color=col, lw=2, ms=5, label=f'R={R} mm/h')
    axes[1].plot(elevations, p_bin,  'o-', color=col, lw=2, ms=5, label=f'R={R} mm/h')

for ax, title in [(axes[0], 'Physics Proxy  p_i = 1 − O_n(θ, R)\n(Ali Abdi + ITU-R P.838)'),
                   (axes[1], 'Binary Proxy (legacy)\nlabel = 1 if θ≥20° and R≤0.5 mm/h')]:
    ax.set_xlabel('Elevation angle θ [degrees]')
    ax.set_ylabel('Label / Probability')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.05, 1.05)
    ax.axvline(20, color='gray', lw=1, linestyle=':', label='θ=20° threshold')

plt.suptitle('Physics-Informed vs Binary Proxy Label', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('proxy_comparison.pdf', bbox_inches='tight')
plt.show()
print("Key insight: physics proxy captures continuous degradation with rain and")
print("elevation — e.g. θ=80°, R=0.6mm/h → physics: p≈0.89, binary: p=0 (wrong)")


## ML Model Training

With the physics-informed label $p_i \in [0,1]$, the ML task changes from **classification** to **regression** — the model learns to estimate the continuous outage-derived probability from forecast features.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def ece_continuous(y_true_p, y_pred_p, n_bins=10):
    """ECE adapted for continuous labels — measures calibration of predicted p
    against physics proxy ground truth."""
    bins = np.linspace(0.0, 1.0, n_bins+1)
    err, n = 0.0, len(y_true_p)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_pred_p >= lo) & (y_pred_p < hi)
        if mask.sum() == 0:
            continue
        err += (mask.sum()/n) * abs(float(np.mean(y_true_p[mask])) -
                                     float(np.mean(y_pred_p[mask])))
    return err


def train_best_regressor(df_train):
    """Train 5 regressors, select best by CV-RMSE, retrain on full set.

    Candidates:
      1. Ridge Regression          — linear baseline
      2. Gradient Boosting Reg.    — selected in SBrT as classifier
      3. Random Forest Reg.        — ensemble alternative
      4. MLP Regressor (64→32)     — neural baseline
      5. SVR (RBF)                 — kernel baseline

    Selection: lowest 5-fold CV RMSE.
    """
    EXCLUDE      = ['pass_id','satellite','start_utc','end_utc','label_success']
    feature_cols = [c for c in df_train.columns if c not in EXCLUDE]
    X = df_train[feature_cols].values
    y = df_train['label_success'].values

    X_tr, X_val, y_tr, y_val = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE)

    imp = ('imputer', SimpleImputer(strategy='median'))
    scl = ('scaler',  StandardScaler())

    candidates = {
        'Ridge':               Pipeline([imp, scl, ('reg', Ridge())]),
        'Gradient Boosting':   Pipeline([imp, ('reg', GradientBoostingRegressor(
                                   n_estimators=200, random_state=RANDOM_STATE))]),
        'Random Forest':       Pipeline([imp, ('reg', RandomForestRegressor(
                                   n_estimators=200, random_state=RANDOM_STATE))]),
        'MLP (64-32)':         Pipeline([imp, scl, ('reg', MLPRegressor(
                                   hidden_layer_sizes=(64,32), max_iter=500,
                                   random_state=RANDOM_STATE))]),
        'SVR (RBF)':           Pipeline([imp, scl, ('reg', SVR(kernel='rbf'))]),
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    print('\n══ Regressor Comparison (5-fold CV — training set) ══')
    print(f"{'Model':<22} {'CV-RMSE':>9} {'Val-RMSE':>9} {'Val-R²':>8} {'Val-ECE':>9}")
    print('─' * 62)

    comparison = {}
    for name, model in candidates.items():
        model.fit(X_tr, y_tr)
        val_p    = np.clip(model.predict(X_val), 0, 1)
        val_rmse = rmse(y_val, val_p)
        val_r2   = r2_score(y_val, val_p)
        val_ece  = ece_continuous(y_val, val_p)
        cv_scores = cross_val_score(model, X, y, cv=kf,
                                    scoring='neg_root_mean_squared_error')
        cv_rmse   = float(-cv_scores.mean())
        print(f'{name:<22} {cv_rmse:>9.4f} {val_rmse:>9.4f} {val_r2:>8.3f} {val_ece:>9.4f}')
        comparison[name] = {'cv_rmse': cv_rmse, 'val_rmse': val_rmse,
                             'r2': val_r2, 'ece': val_ece}

    best_name = min(comparison, key=lambda k: comparison[k]['cv_rmse'])
    best_info = comparison[best_name]
    print(f'\n✅ Best model: {best_name}  (CV-RMSE = {best_info["cv_rmse"]:.4f})')

    # Retrain on full training set
    best_reg = candidates[best_name]
    best_reg.fit(X, y)
    return best_reg, feature_cols, best_name, comparison

print("✅ ML regression functions loaded")


## MILP Scheduling + Baselines

In [ ]:
def _has_conflict(start_a, end_a, start_b, end_b, setup_td):
    return (pd.to_datetime(start_a)-setup_td < pd.to_datetime(end_b)+setup_td and
            pd.to_datetime(start_b)-setup_td < pd.to_datetime(end_a)+setup_td)

def _conflicts_with_any(row, scheduled, setup_td):
    s_new = pd.to_datetime(row['start_utc'])
    e_new = pd.to_datetime(row['end_utc'])
    window = pd.Timedelta(hours=2)
    for c in scheduled:
        s_c = pd.to_datetime(c['start_utc'])
        if s_c > e_new + setup_td + window: break
        if _has_conflict(s_new, e_new, c['start_utc'], c['end_utc'], setup_td):
            return True
    return False

def schedule_milp(passes_df, success_probs,
                  bitrate_kbps=BITRATE_KBPS, min_setup_s=MIN_SETUP_S,
                  setup_cost_kb=SETUP_COST_KB, max_total_s=MAX_TOTAL_SCHEDULED_S):
    import time as _time
    prob = pulp.LpProblem('schedule', pulp.LpMaximize)
    n    = len(passes_df)
    x    = [pulp.LpVariable(f'x_{i}', cat='Binary') for i in range(n)]
    vals = [passes_df.iloc[i]['duration_s'] * float(success_probs[i]) * bitrate_kbps
            for i in range(n)]
    prob += pulp.lpSum([vals[i]*x[i] for i in range(n)]) - setup_cost_kb*pulp.lpSum(x)
    starts   = pd.to_datetime(passes_df['start_utc'])
    ends     = pd.to_datetime(passes_df['end_utc'])
    setup_td = pd.Timedelta(seconds=int(min_setup_s))
    for i in range(n):
        si, ei = starts.iloc[i]-setup_td, ends.iloc[i]+setup_td
        for j in range(i+1, n):
            if starts.iloc[j]-setup_td > ei: break
            if si < ends.iloc[j]+setup_td:
                prob += x[i]+x[j] <= 1
    if max_total_s is not None:
        prob += pulp.lpSum([passes_df.iloc[i]['duration_s']*x[i]
                            for i in range(n)]) <= float(max_total_s)
    _n_c = len(prob.constraints)
    _t0  = _time.time()
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    schedule_milp.last_stats = {
        'n_passes': n, 'n_variables': n, 'n_constraints': _n_c,
        'solve_time_s': _time.time()-_t0, 'status': pulp.LpStatus[prob.status]
    }
    return [bool(pulp.value(x[i]) is not None and pulp.value(x[i]) > 0.5)
            for i in range(n)]

def greedy_by_start(passes_df, min_setup_s=MIN_SETUP_S):
    setup_td  = pd.Timedelta(seconds=int(min_setup_s))
    df        = passes_df.sort_values('start_utc').reset_index(drop=True)
    scheduled, chosen = [], set()
    for _, row in df.iterrows():
        if not _conflicts_with_any(row, scheduled, setup_td):
            scheduled.append(row); chosen.add(row['pass_id'])
    return chosen

def greedy_by_expected_value(passes_df, exp_vals, min_setup_s=MIN_SETUP_S):
    order    = np.argsort(-np.array(exp_vals))
    setup_td = pd.Timedelta(seconds=int(min_setup_s))
    chosen, sched = set(), []
    for idx in order:
        if exp_vals[idx] <= 0: continue
        row = passes_df.iloc[idx]
        if not _conflicts_with_any(row, sched, setup_td):
            sched.append(row); chosen.add(row['pass_id'])
    return chosen

def satnogs_policy(passes_df, min_elevation_deg=20.0, min_setup_s=MIN_SETUP_S):
    setup_td   = pd.Timedelta(seconds=int(min_setup_s))
    candidates = (passes_df[passes_df['max_elevation_deg'] >= min_elevation_deg]
                  .sort_values('start_utc').reset_index(drop=True))
    scheduled, chosen = [], set()
    for _, row in candidates.iterrows():
        if not _conflicts_with_any(row, scheduled, setup_td):
            scheduled.append(row); chosen.add(row['pass_id'])
    return chosen

def apply_capacity_cap(ids_set, passes_df, max_s=MAX_TOTAL_SCHEDULED_S):
    if max_s is None: return ids_set
    ordered = passes_df[passes_df['pass_id'].isin(ids_set)].sort_values('start_utc')
    total, kept = 0.0, set()
    for _, row in ordered.iterrows():
        if total + row['duration_s'] <= max_s:
            kept.add(row['pass_id']); total += row['duration_s']
        else: break
    return kept

def count_conflicts(ids_set, passes_df, min_setup_s=MIN_SETUP_S):
    setup_td = pd.Timedelta(seconds=int(min_setup_s))
    rows = (passes_df[passes_df['pass_id'].isin(ids_set)]
            .sort_values('start_utc').reset_index(drop=True))
    c = 0
    for i in range(len(rows)):
        for j in range(i+1, len(rows)):
            if (pd.to_datetime(rows.iloc[j]['start_utc'])-setup_td >
                    pd.to_datetime(rows.iloc[i]['end_utc'])+setup_td): break
            if _has_conflict(rows.iloc[i]['start_utc'], rows.iloc[i]['end_utc'],
                             rows.iloc[j]['start_utc'], rows.iloc[j]['end_utc'], setup_td):
                c += 1
    return c

def evaluate_schedule(passes_df, sel_ids, true_p, pred_p,
                       bitrate_kbps=BITRATE_KBPS, min_setup_s=MIN_SETUP_S):
    """Evaluate schedule using CONTINUOUS true_p (physics proxy from ERA5).

    true_p = p_ERA5 = 1 - O_n(theta, R_ERA5)  ← ground truth
    pred_p = model predictions                  ← used for expected throughput
    """
    idx_map = {passes_df.iloc[i]['pass_id']: i for i in range(len(passes_df))}
    n_conf  = count_conflicts(sel_ids, passes_df, min_setup_s)
    exp = rea = 0.0
    for p in sel_ids:
        if p not in idx_map: continue
        i   = idx_map[p]
        dur = passes_df.iloc[i]['duration_s']
        exp += dur * bitrate_kbps * float(pred_p[i])
        rea += dur * bitrate_kbps * float(true_p[i])
    return {
        'expected_mb':   exp / 8e6,
        'realized_mb':   rea / 8e6,
        'n_passes':      len([p for p in sel_ids if p in idx_map]),
        'n_conflicts':   n_conf,
        'conflict_flag': '⚠️' if n_conf > 0 else '✅',
    }

def print_schedule_table(results_dict, title=''):
    if title: print(f'\n{title}')
    print(f"{'Method':<24} {'Expected (MB)':>14} {'Realized (MB)':>15} {'Passes':>7}")
    print('─' * 63)
    for k, v in results_dict.items():
        print(f"{k:<24} {v['expected_mb']:>14.1f} {v['realized_mb']:>15.1f} {v['n_passes']:>7}")

print("✅ MILP + baseline functions loaded")


## Main Execution

### Pipeline
```
TLE → passes → Open-Meteo forecast → features
                                      ↓
                                 physics_proxy_label (training)
                                      ↓
                                 Regressor training
                                      ↓
                                 pred_p = model.predict(X_test)

ERA5 archive → p_ERA5 = physics_proxy_label (test ground truth)
```

In [ ]:
print(f'Loading {N_SATELLITES} Starlink satellites...')
sat_list = get_satellites()

now         = datetime.utcnow()
train_end   = now - timedelta(days=HISTORY_DAYS - TRAIN_DAYS)
train_start = train_end - timedelta(days=TRAIN_DAYS)
test_start  = train_end
test_end    = now - timedelta(days=3)

print(f'Train : {train_start.date()} → {train_end.date()} ({TRAIN_DAYS} days)')
print(f'Test  : {test_start.date()} → {test_end.date()}  ({(test_end-test_start).days} days)')

# ── Training passes ────────────────────────────────────────────────────────────
_tr_cache = _cache_path('passes_train_physics')
if _cache_exists('passes_train_physics'):
    print('\nTrain passes: loading cache...')
    passes_train = pd.read_csv(_tr_cache, parse_dates=['start_utc','end_utc'])
else:
    print(f'\nComputing train passes ({N_SATELLITES} satellites)...')
    passes_train = compute_passes_all_sats(sat_list, STATION, train_start, train_end)
    passes_train.to_csv(_tr_cache, index=False)
print(f'  {len(passes_train)} passes | {passes_train["satellite"].nunique()} satellites')

# Features + physics labels (training — with noise)
meteo_tr = fetch_open_meteo_forecast(STATION['lat'], STATION['lon'],
                                     train_start - timedelta(hours=1),
                                     train_end   + timedelta(hours=1))
df_train = build_features_labels(passes_train, meteo_tr,
                                  noise_sigma=LABEL_FLIP_SIGMA,
                                  prefix='tr', use_physics=True)
print(f'  Physics labels — mean={df_train["label_success"].mean():.3f}  '
      f'std={df_train["label_success"].std():.3f}')

# ── Train model ────────────────────────────────────────────────────────────────
print('\nTraining and selecting best regressor...')
reg, feature_cols, best_model_name, model_comparison = train_best_regressor(df_train)


In [ ]:
# ── Test passes ────────────────────────────────────────────────────────────────
_te_cache = _cache_path('passes_test_physics')
if _cache_exists('passes_test_physics'):
    print('Test passes: loading cache...')
    passes_test = pd.read_csv(_te_cache, parse_dates=['start_utc','end_utc'])
else:
    print(f'Computing test passes ({N_SATELLITES} satellites)...')
    passes_test = compute_passes_all_sats(sat_list, STATION, test_start, test_end)
    passes_test.to_csv(_te_cache, index=False)
print(f'  {len(passes_test)} passes | {passes_test["satellite"].nunique()} satellites')

# ── Count conflicts ────────────────────────────────────────────────────────────
setup_td  = pd.Timedelta(seconds=MIN_SETUP_S)
starts_te = pd.to_datetime(passes_test['start_utc'])
ends_te   = pd.to_datetime(passes_test['end_utc'])
n_conf_total = 0
for i in range(len(passes_test)):
    for j in range(i+1, len(passes_test)):
        if starts_te.iloc[j]-setup_td > ends_te.iloc[i]+setup_td: break
        if _has_conflict(starts_te.iloc[i], ends_te.iloc[i],
                         starts_te.iloc[j], ends_te.iloc[j], setup_td):
            n_conf_total += 1
print(f'  Conflicting pairs: {n_conf_total}')

# ── Features from forecast (as in production) ─────────────────────────────────
meteo_fc  = fetch_open_meteo_forecast(STATION['lat'], STATION['lon'],
                                      test_start - timedelta(hours=1),
                                      test_end   + timedelta(hours=1))
df_test   = build_features_labels(passes_test, meteo_fc, noise_sigma=0.0,
                                   prefix='te', use_physics=True)
X_test    = df_test[feature_cols].values

# ── Ground truth from ERA5 archive (INDEPENDENT source) ───────────────────────
print('  Fetching ERA5 archive (ground truth)...')
meteo_arch = fetch_open_meteo_archive(STATION['lat'], STATION['lon'],
                                      test_start - timedelta(hours=1),
                                      test_end   + timedelta(hours=1))

# Physics proxy labels from ERA5 (no noise — deterministic ground truth)
true_p_era5 = np.array([
    physics_proxy_label(
        features_from_pass(row, meteo_arch)['max_elevation_deg'],
        features_from_pass(row, meteo_arch)['precipitation_mean'],
        noise_sigma=0.0
    )
    for _, row in passes_test.iterrows()
])

# Model predictions (from forecast features)
pred_p = np.clip(reg.predict(X_test), 0.0, 1.0)

# Forecast-based physics proxy (for concordance)
pred_p_phys = np.array([
    physics_proxy_label(
        features_from_pass(row, meteo_fc)['max_elevation_deg'],
        features_from_pass(row, meteo_fc)['precipitation_mean'],
        noise_sigma=0.0
    )
    for _, row in passes_test.iterrows()
])

# ── Diagnostics ────────────────────────────────────────────────────────────────
print('\n══ Test Set Diagnostics ══')
print(f'  true_p (ERA5):   mean={true_p_era5.mean():.3f}  std={true_p_era5.std():.3f}')
print(f'  pred_p (model):  mean={pred_p.mean():.3f}  std={pred_p.std():.3f}')
print(f'  physics_fc:      mean={pred_p_phys.mean():.3f}  std={pred_p_phys.std():.3f}')
test_rmse = rmse(true_p_era5, pred_p)
test_r2   = r2_score(true_p_era5, pred_p)
test_ece  = ece_continuous(true_p_era5, pred_p)
conc = np.corrcoef(true_p_era5, pred_p)[0,1]
print(f'\n  Model vs ERA5:  RMSE={test_rmse:.4f}  R²={test_r2:.3f}  '
      f'ECE={test_ece:.4f}  Pearson r={conc:.3f}')

# AUC (binarize at 0.5 for comparison)
try:
    auc_test = roc_auc_score((true_p_era5 >= 0.5).astype(int), pred_p)
    print(f'  AUC (threshold 0.5): {auc_test:.3f}')
except:
    pass


In [ ]:
# ── Scheduling ────────────────────────────────────────────────────────────────
print('\nOptimizing schedule...')
exp_vals = [passes_test.iloc[i]['duration_s'] * pred_p[i] * BITRATE_KBPS
            for i in range(len(passes_test))]

ids_milp    = [passes_test.iloc[i]['pass_id']
               for i, c in enumerate(schedule_milp(passes_test, pred_p,
                   setup_cost_kb=SETUP_COST_KB, max_total_s=MAX_TOTAL_SCHEDULED_S))
               if c]
s = schedule_milp.last_stats
print(f"\n══ MILP Instance & Solver Metrics ══")
print(f"  Candidate passes  : {s['n_passes']}")
print(f"  Binary variables  : {s['n_variables']}")
print(f"  Conflict constrs  : {s['n_constraints']}")
print(f"  Solver time       : {s['solve_time_s']:.3f} s")
print(f"  Status            : {s['status']}")

ids_gstart  = apply_capacity_cap(greedy_by_start(passes_test),                   passes_test)
ids_gvalue  = apply_capacity_cap(greedy_by_expected_value(passes_test, exp_vals), passes_test)
ids_satnogs = apply_capacity_cap(satnogs_policy(passes_test),                     passes_test)

# Evaluate using CONTINUOUS true_p from ERA5
results = {
    'Predictive MILP': evaluate_schedule(passes_test, ids_milp,    true_p_era5, pred_p),
    'Greedy Start':    evaluate_schedule(passes_test, ids_gstart,  true_p_era5, pred_p),
    'Greedy Value':    evaluate_schedule(passes_test, ids_gvalue,  true_p_era5, pred_p),
    'SatNOGS-like':    evaluate_schedule(passes_test, ids_satnogs, true_p_era5, pred_p),
}

print_schedule_table(results, title='\n══ Scheduling Results (ground truth = physics proxy ERA5) ══')

baseline = results['Greedy Start']['realized_mb'] + 1e-9
for k, v in results.items():
    if k != 'Greedy Start':
        g = 100*(v['realized_mb'] - baseline)/baseline
        print(f"  MILP gain vs Greedy Start ({k}): {g:+.1f}%")

print(f'\nConflicting pairs in test set: {n_conf_total}')
if all(v['n_conflicts'] == 0 for v in results.values()):
    print('✅ All methods conflict-free — fair comparison')


In [ ]:
# ══ Calibration Analysis ══════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: predicted vs ERA5 physics proxy
axes[0].scatter(true_p_era5, pred_p, alpha=0.15, s=5, color='#3B8BD4')
axes[0].plot([0,1],[0,1],'k--', lw=1, label='Perfect')
axes[0].set_xlabel('True p (ERA5 physics proxy)')
axes[0].set_ylabel('Predicted p (model)')
axes[0].set_title(f'Predicted vs True\nR²={test_r2:.3f}  RMSE={test_rmse:.4f}')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Reliability diagram (binned)
bins = np.linspace(0, 1, 11)
bin_centers  = (bins[:-1] + bins[1:]) / 2
mean_pred_p  = []
mean_true_p  = []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (pred_p >= lo) & (pred_p < hi)
    if mask.sum() > 0:
        mean_pred_p.append(np.mean(pred_p[mask]))
        mean_true_p.append(np.mean(true_p_era5[mask]))
axes[1].plot([0,1],[0,1],'k--', lw=1, label='Perfect calibration')
axes[1].plot(mean_pred_p, mean_true_p, 'o-', color='#1D9E75', lw=2, ms=7,
             label=f'Model  ECE={test_ece:.4f}')
axes[1].set_xlabel('Mean predicted p (bins)')
axes[1].set_ylabel('Mean true p (ERA5)')
axes[1].set_title('Reliability Diagram\n(continuous label calibration)')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)

plt.suptitle(f'Model Calibration — {best_model_name}\n'
             f'Physics proxy labels (Ali Abdi + ITU-R P.838)',
             fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig('calibration_physics.pdf', bbox_inches='tight')
plt.show()
print("Saved: calibration_physics.pdf")

# Distribution of predicted p_i
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pred_p,       bins=30, alpha=0.6, color='#3B8BD4', label='Predicted p (model)')
ax.hist(true_p_era5,  bins=30, alpha=0.6, color='#1D9E75', label='True p (ERA5 physics)')
ax.hist(pred_p_phys,  bins=30, alpha=0.4, color='#E24B4A', label='Physics p (forecast)')
ax.set_xlabel('p_i value'); ax.set_ylabel('Count')
ax.set_title('Distribution of p_i — Physics proxy vs Model predictions')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('p_distribution.pdf', bbox_inches='tight')
plt.show()
print("Saved: p_distribution.pdf")


## Comparison: Physics Proxy vs Binary Proxy

Key journal contribution: show that the physics-informed label improves scheduling quality over the binary threshold rule.

In [ ]:
# ── Binary proxy labels for comparison ────────────────────────────────────────
df_train_bin = build_features_labels(passes_train, meteo_tr, noise_sigma=0.05,
                                      prefix='tr_bin', use_physics=False)
df_test_bin  = build_features_labels(passes_test, meteo_fc,  noise_sigma=0.0,
                                      prefix='te_bin', use_physics=False)

# Train classifier on binary labels
from sklearn.ensemble import GradientBoostingClassifier
clf_bin = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf',     GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE))
])
clf_bin.fit(df_train_bin[feature_cols].values,
            df_train_bin['label_success'].values)
pred_p_bin = clf_bin.predict_proba(X_test)[:, 1]

# True labels from ERA5 — binary version for AUC
true_bin_era5 = (true_p_era5 >= 0.5).astype(int)

print("══ Comparison: Physics Proxy vs Binary Proxy ══")
print(f"{'Metric':<30} {'Physics Proxy':>15} {'Binary Proxy':>14}")
print('─' * 62)

# RMSE vs ERA5 physics proxy
rmse_phys = rmse(true_p_era5, pred_p)
rmse_bin  = rmse(true_p_era5, pred_p_bin)
print(f"{'RMSE (vs ERA5 physics)':<30} {rmse_phys:>15.4f} {rmse_bin:>14.4f}")

# ECE
ece_phys = ece_continuous(true_p_era5, pred_p)
ece_bin  = ece_continuous(true_p_era5, pred_p_bin)
print(f"{'ECE (vs ERA5 physics)':<30} {ece_phys:>15.4f} {ece_bin:>14.4f}")

# AUC (binarized)
try:
    auc_phys = roc_auc_score(true_bin_era5, pred_p)
    auc_bin  = roc_auc_score(true_bin_era5, pred_p_bin)
    print(f"{'AUC (binary threshold 0.5)':<30} {auc_phys:>15.3f} {auc_bin:>14.3f}")
except: pass

# Scheduling comparison
ids_milp_bin = [passes_test.iloc[i]['pass_id']
                for i, c in enumerate(schedule_milp(passes_test, pred_p_bin,
                    setup_cost_kb=SETUP_COST_KB, max_total_s=MAX_TOTAL_SCHEDULED_S))
                if c]
exp_vals_bin = [passes_test.iloc[i]['duration_s'] * pred_p_bin[i] * BITRATE_KBPS
                for i in range(len(passes_test))]
ids_gstart_bin = apply_capacity_cap(greedy_by_start(passes_test), passes_test)

r_milp_phys = evaluate_schedule(passes_test, ids_milp,     true_p_era5, pred_p)
r_milp_bin  = evaluate_schedule(passes_test, ids_milp_bin, true_p_era5, pred_p_bin)
r_gs        = evaluate_schedule(passes_test, ids_gstart,   true_p_era5, pred_p)

print(f"\n{'MILP Realized MB (physics)':<30} {r_milp_phys['realized_mb']:>15.1f}")
print(f"{'MILP Realized MB (binary)':<30} {r_milp_bin['realized_mb']:>14.1f}")
print(f"{'Greedy Start Realized MB':<30} {r_gs['realized_mb']:>15.1f}")
gain_phys = 100*(r_milp_phys['realized_mb']-r_gs['realized_mb'])/(r_gs['realized_mb']+1e-9)
gain_bin  = 100*(r_milp_bin['realized_mb'] -r_gs['realized_mb'])/(r_gs['realized_mb']+1e-9)
print(f"\n  MILP gain (physics proxy): {gain_phys:+.1f}%")
print(f"  MILP gain (binary proxy):  {gain_bin:+.1f}%")
delta = r_milp_phys['realized_mb'] - r_milp_bin['realized_mb']
print(f"  Delta (physics − binary):  {delta:+.1f} MB  ({100*delta/(r_milp_bin['realized_mb']+1e-9):+.1f}%)")


## Zero-Shot Generalization (10 GS, 6 continents)

In [ ]:
results_gen = []
print(f'Model: {best_model_name} — applied without retraining to {len(STATIONS_GLOBAL)} stations\n')

for gs in STATIONS_GLOBAL:
    print(f'── {gs["name"]} ({gs["continent"]}) ──')
    try:
        key  = (f'passes_gen_phys_{gs["name"].replace(" ","_").replace("(","").replace(")","")}'
                .replace(',',''))
        path = _cache_path(key)
        if _cache_exists(key):
            passes_gs = pd.read_csv(path, parse_dates=['start_utc','end_utc'])
        else:
            passes_gs = compute_passes_all_sats(sat_list, gs, test_start, test_end)
            passes_gs.to_csv(path, index=False)

        if len(passes_gs) < 10:
            print(f'  ⚠️  Only {len(passes_gs)} passes — skipping'); continue

        meteo_fc_gs   = fetch_open_meteo_forecast(gs['lat'], gs['lon'],
                            test_start-timedelta(hours=1), test_end+timedelta(hours=1))
        meteo_arch_gs = fetch_open_meteo_archive(gs['lat'], gs['lon'],
                            test_start-timedelta(hours=1), test_end+timedelta(hours=1))

        df_gs  = build_features_labels(passes_gs, meteo_fc_gs, 0.0, 'gs', use_physics=True)
        X_gs   = df_gs[feature_cols].values
        pred_gs = np.clip(reg.predict(X_gs), 0, 1)

        true_gs = np.array([
            physics_proxy_label(
                features_from_pass(row, meteo_arch_gs)['max_elevation_deg'],
                features_from_pass(row, meteo_arch_gs)['precipitation_mean'],
                noise_sigma=0.0)
            for _, row in passes_gs.iterrows()
        ])

        rmse_gs = rmse(true_gs, pred_gs)
        r2_gs   = r2_score(true_gs, pred_gs)
        ece_gs  = ece_continuous(true_gs, pred_gs)
        try:
            auc_gs = roc_auc_score((true_gs>=0.5).astype(int), pred_gs)
        except:
            auc_gs = float('nan')

        print(f'  Passes={len(passes_gs)}  AUC={auc_gs:.3f}  '
              f'RMSE={rmse_gs:.4f}  R²={r2_gs:.3f}  ECE={ece_gs:.4f}')

        results_gen.append({
            'GS': gs['name'], 'Continent': gs['continent'],
            'Passes': len(passes_gs),
            'AUC':  round(auc_gs, 3), 'RMSE': round(rmse_gs, 4),
            'R2':   round(r2_gs, 3),  'ECE':  round(ece_gs, 4),
            'is_train': gs['name'] == STATIONS_GLOBAL[0]['name']
        })
    except Exception as e:
        print(f'  ❌ Error: {e}')

df_gen = pd.DataFrame(results_gen)
print('\n══ Generalization Table ══')
print(df_gen[['GS','Passes','AUC','RMSE','R2','ECE']].to_string(index=False))

ext = df_gen[~df_gen['is_train']]
if len(ext) > 0:
    print(f'\nZero-shot variation (external GS):')
    print(f'  AUC : {ext["AUC"].mean():.3f} ± {ext["AUC"].std():.3f}')
    print(f'  RMSE: {ext["RMSE"].mean():.4f} ± {ext["RMSE"].std():.4f}')
    print(f'  ECE : {ext["ECE"].mean():.4f} ± {ext["ECE"].std():.4f}')

# Plot
from matplotlib.patches import Patch
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#1D9E75' if r else '#3B8BD4' for r in df_gen['is_train']]
names  = [r.split('(')[0].strip() for r in df_gen['GS']]
for ax, col, title in [
    (axes[0], 'AUC',  'AUC  (higher = better)'),
    (axes[1], 'RMSE', 'RMSE  (lower = better)'),
    (axes[2], 'ECE',  'ECE   (lower = better)'),
]:
    vals = df_gen[col].values
    ax.barh(range(len(df_gen)), vals, color=colors, alpha=0.85)
    ax.set_yticks(range(len(df_gen))); ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel(col); ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3, axis='x'); ax.invert_yaxis()
    for i, v in enumerate(vals):
        ax.text(v+max(vals)*0.01, i, f'{v:.3f}', va='center', fontsize=8,
                fontweight='bold' if df_gen.iloc[i]['is_train'] else 'normal')
axes[0].legend(handles=[Patch(color='#1D9E75',label='Training GS'),
                          Patch(color='#3B8BD4',label='Zero-shot')], fontsize=8)
plt.suptitle(f'Zero-Shot Generalization — {best_model_name} — Physics Proxy Labels',
             fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig('generalization_physics.pdf', bbox_inches='tight')
plt.show()
print('Saved: generalization_physics.pdf')


## Sensitivity Analysis — ε and Label Noise

In [ ]:
# ── Sensitivity to ε ──────────────────────────────────────────────────────────
print('══ Sensitivity to ε (C_setup = 0) ══')
epsilon_vals = [None, 7200, 3600, 1800]

for eps in epsilon_vals:
    r = {}
    eps_str = 'no limit' if eps is None else f'{eps} s ({eps//60} min)'
    tag = '  ◄ current' if eps == MAX_TOTAL_SCHEDULED_S else ''

    chosen   = schedule_milp(passes_test, pred_p, setup_cost_kb=0.0, max_total_s=eps)
    ids_m    = [passes_test.iloc[i]['pass_id'] for i, c in enumerate(chosen) if c]
    ids_gs   = apply_capacity_cap(greedy_by_start(passes_test), passes_test, eps)
    ids_gv   = apply_capacity_cap(greedy_by_expected_value(passes_test, exp_vals),
                                   passes_test, eps)
    ids_sn   = apply_capacity_cap(satnogs_policy(passes_test), passes_test, eps)

    rr = {
        'Predictive MILP': evaluate_schedule(passes_test, ids_m,  true_p_era5, pred_p),
        'Greedy Start':    evaluate_schedule(passes_test, ids_gs, true_p_era5, pred_p),
        'Greedy Value':    evaluate_schedule(passes_test, ids_gv, true_p_era5, pred_p),
        'SatNOGS-like':    evaluate_schedule(passes_test, ids_sn, true_p_era5, pred_p),
    }
    gain = 100*(rr['Predictive MILP']['realized_mb'] -
                rr['Greedy Start']['realized_mb']) / (rr['Greedy Start']['realized_mb']+1e-9)
    print_schedule_table(rr, title=f'\nε = {eps_str}{tag}')
    print(f'  → MILP gain vs Greedy Start: {gain:+.1f}%')

# ── Sensitivity to label noise (noise_sigma) ──────────────────────────────────
print('\n══ Impact of Label Noise (noise_sigma) ══')
noise_vals = [0.0, 0.02, 0.05, 0.10, 0.20]
EXCLUDE = ['pass_id','satellite','start_utc','end_utc','label_success']

print(f"{'noise_sigma':>12} {'RMSE':>8} {'ECE':>8} {'AUC':>7} {'MILP MB':>9} {'GS MB':>8} {'Gain':>7}")
print('─' * 65)

rows_noise = []
for ns in noise_vals:
    df_ns = build_features_labels(passes_train, meteo_tr, noise_sigma=ns,
                                   prefix='ns', use_physics=True)
    _fc   = [c for c in df_ns.columns if c not in EXCLUDE]
    reg_ns = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('reg',     GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE))
    ])
    reg_ns.fit(df_ns[_fc].values, df_ns['label_success'].values)
    p_ns = np.clip(reg_ns.predict(X_test), 0, 1)

    rm_ns  = rmse(true_p_era5, p_ns)
    ec_ns  = ece_continuous(true_p_era5, p_ns)
    try:
        au_ns = roc_auc_score((true_p_era5>=0.5).astype(int), p_ns)
    except:
        au_ns = float('nan')

    chosen_ns = schedule_milp(passes_test, p_ns, setup_cost_kb=0.0, max_total_s=None)
    ids_ns    = [passes_test.iloc[i]['pass_id'] for i, c in enumerate(chosen_ns) if c]
    ids_gs_ns = apply_capacity_cap(greedy_by_start(passes_test), passes_test, None)
    r_ns  = evaluate_schedule(passes_test, ids_ns,    true_p_era5, p_ns)
    r_gs_ = evaluate_schedule(passes_test, ids_gs_ns, true_p_era5, p_ns)
    gain_ns = 100*(r_ns['realized_mb']-r_gs_['realized_mb'])/(r_gs_['realized_mb']+1e-9)
    tag = '  ◄ current' if ns == LABEL_FLIP_SIGMA else ''
    print(f'{ns:>12.2f} {rm_ns:>8.4f} {ec_ns:>8.4f} {au_ns:>7.3f} '
          f'{r_ns["realized_mb"]:>9.1f} {r_gs_["realized_mb"]:>8.1f} {gain_ns:>+6.1f}%{tag}')
    rows_noise.append({'sigma': ns, 'RMSE': rm_ns, 'ECE': ec_ns,
                       'AUC': au_ns, 'MILP': r_ns['realized_mb'], 'Gain': gain_ns})


## Summary for Journal Paper

In [ ]:
print('═'*65)
print('JOURNAL PAPER — SUMMARY OF RESULTS')
print('═'*65)
print(f'\nPhysics-Informed Proxy Label: p_i = 1 − O_n(θ_i, R_i, f_c)')
print(f'  Channel model : Ali Abdi et al. (Rice shadowed, elevation-dependent)')
print(f'  Rain atten.   : ITU-R P.838 (Ku-band {FC_GHZ} GHz, k={ITU_P838_KR}, α={ITU_P838_AR})')
print(f'  Rain height   : h_R = {RAIN_HEIGHT_KM} km (ITU-R P.839)')
print(f'  Link budget   : P_t={PS_DBM} dBm, G_i={DI_DB} dB, G_n={DN_DB} dB, SNR_th={SNR_TH_DB} dB')
print(f'\nML Model: {best_model_name}')
print(f'  Task          : Regression (p_i ∈ [0,1], continuous label)')
print(f'  Train labels  : Physics proxy from Open-Meteo FORECAST + noise σ={LABEL_FLIP_SIGMA}')
print(f'  Test GT       : Physics proxy from ERA5 ARCHIVE (independent source)')
print(f'  CV-RMSE       : {model_comparison[best_model_name]["cv_rmse"]:.4f}')
print(f'  Test RMSE     : {test_rmse:.4f}')
print(f'  Test R²       : {test_r2:.3f}')
print(f'  Test ECE      : {test_ece:.4f}')
print(f'\nScheduling ({N_SATELLITES} Starlink satellites, {len(passes_test)} passes, ε={MAX_TOTAL_SCHEDULED_S}s):')
r_milp = results['Predictive MILP']
r_gs_m = results['Greedy Start']
g_pct  = 100*(r_milp["realized_mb"]-r_gs_m["realized_mb"])/(r_gs_m["realized_mb"]+1e-9)
print(f'  Realized throughput (MILP)         : {r_milp["realized_mb"]:.1f} MB')
print(f'  Realized throughput (Greedy Start) : {r_gs_m["realized_mb"]:.1f} MB')
print(f'  MILP gain vs Greedy Start          : {g_pct:+.1f}%')
print(f'  Solver time                        : {s["solve_time_s"]:.3f} s')
print(f'\nGeneralization (zero-shot, {len(ext)} external GS):')
print(f'  Mean AUC  : {ext["AUC"].mean():.3f} ± {ext["AUC"].std():.3f}')
print(f'  Mean RMSE : {ext["RMSE"].mean():.4f} ± {ext["RMSE"].std():.4f}')
print(f'  Mean ECE  : {ext["ECE"].mean():.4f} ± {ext["ECE"].std():.4f}')
print(f'\nKey advantage vs binary proxy (SBrT):')
print(f'  Binary label ignores elevation effect on rain path length.')
print(f'  Physics label: θ=80°, R=0.6mm/h → p=0.89 (binary=0, physically wrong)')
print(f'  Physics label: θ=21°, R=0.4mm/h → p=0.73 (binary=1, risk captured)')
print('═'*65)
